# Data preparation: data types

This notebook prepares `customer_train.csv` (19,158 students, 14 columns) for our analysis. It does three things:

1. Change every column to a data type that fits its content and needs less memory
2. Check that nothing went wrong while converting
3. Look for data problems like logical breaks, duplicates and outliers (only counted, nothing gets changed)

At the end the converted data is saved as a new file in `week 2`. The original `customer_train.csv` is only read, never changed.

Run the cells from top to bottom (or "Run All"), because every cell uses results from the cells above it.

## 1. Loading the data

pandas reads the CSV file into a **DataFrame**. That's basically a table: one row per student, one column per variable.

The CSV file is one folder above this notebook, that's why the path starts with `../`.

In [1]:
import pandas as pd

df = pd.read_csv("../customer_train.csv")
df.head()

,student_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,job_change
0,8949,city_103,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevant experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevant experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0
3,33241,city_115,0.789,NaN,No relevant experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1.0
4,666,city_162,0.767,Male,Has relevant experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0.0


Every column has a **data type** (dtype). It tells pandas what kind of values are in the column and how much memory each value needs. When reading a CSV, pandas has to guess the type:

- whole numbers become `int64`
- numbers with decimals become `float64`
- everything else becomes `object`, which here simply means text

These guesses work, but they're rarely the best choice.

In [2]:
print("Old data types:")
print(df.dtypes)

Old data types:
student_id                  int64
city                       object
city_development_index    float64
gender                     object
relevant_experience        object
enrolled_university        object
education_level            object
major_discipline           object
experience                 object
company_size               object
company_type               object
last_new_job               object
training_hours              int64
job_change                float64
dtype: object


Before changing anything we write down two numbers: how many values are missing in each column, and how much memory the whole table needs. Later we compare against them to see if the conversion lost any values and how much memory we saved.

In [3]:
missing_before = df.isna().sum()
memory_before = df.memory_usage(deep=True).sum()

print(f"Missing values: {missing_before.sum():,}")
print(f"Memory: {memory_before / 1e6:.1f} MB")

Missing values: 20,733
Memory: 11.0 MB


## 2. Why change the data types?

Our first goal is to store the data as efficiently as possible. The right data type helps in two ways.

**Less memory.** A text column (`object`) stores the full text again in every row, so "Has relevant experience" is stored 19,158 times. The other types are much smaller:

| Type | What it stores | Memory per row |
|---|---|---|
| object | the full text, once per row | about 50 to 70 bytes in our data |
| category | every text only once, each row just holds a small number pointing to it | 1 byte |
| bool | True or False | 1 byte |
| int64 / int32 | whole numbers | 8 / 4 bytes |
| float64 / float16 | decimal numbers | 8 / 2 bytes |

**Correct meaning.** The type also decides what you can do with a column. Text is compared letter by letter, so `"10000+"` comes before `"50-99"` because "1" comes before "5". With an ordered category pandas knows the real order and can sort and filter correctly.

To pick the type we look at the **measurement level** of each column:

- **Nominal:** categories without an order, e.g. city or gender
- **Ordinal:** categories with a natural order, e.g. education level
- **Binary:** only two possible values, e.g. yes / no
- **Numeric:** real numbers you can calculate with, e.g. training hours

### Target types

| Column | Measurement level | Now | Target | Notes |
|---|---|---|---|---|
| student_id | nominal (identifier) | int64 | int32 | |
| city | nominal | object | category | keep the labels, the city numbers mean nothing |
| city_development_index | numeric | float64 | float16 | values have at most 3 decimals, no rounding needed |
| gender | nominal | object | category | |
| relevant_experience | binary | object | bool | needs a comparison, see section 4 |
| enrolled_university | ordinal | object | ordered category | see section 5 |
| education_level | ordinal | object | ordered category | see section 5 |
| major_discipline | nominal | object | category | |
| experience | ordinal | object | ordered category | see section 5 |
| company_size | ordinal | object | ordered category | see section 5 |
| company_type | nominal | object | category | |
| last_new_job | ordinal | object | ordered category | see section 5, renaming it is a later step |
| training_hours | numeric | int64 | int32 | |
| job_change | binary (target) | float64 | bool | |

`int32` still holds numbers up to about 2 billion, more than enough for IDs up to 33,380 and at most 336 training hours.

`float16` is less precise: 0.92 is stored as 0.9199. That difference is far too small to matter for our analysis.

Missing values stay empty (NaN) for now. Filling them, e.g. with "Unknown", is part of the missing data step.

## 3. Converting the simple columns

`astype` changes the type of a column. We give it a dictionary: column name on the left, new type on the right. All columns where a plain conversion works are done in one go.

`job_change` only contains 0.0 and 1.0 and has no missing values, so `"bool"` simply turns 0 into False and 1 into True.

In [4]:
df = df.astype({
    "student_id": "int32",
    "city": "category",
    "city_development_index": "float16",
    "gender": "category",
    "major_discipline": "category",
    "company_type": "category",
    "training_hours": "int32",
    "job_change": "bool",
})

## 4. relevant_experience: why astype doesn't work here

The column contains the texts "Has relevant experience" and "No relevant experience". It looks like `astype("bool")` should work, but Python treats **every text that isn't empty as True**. The small example shows the problem:

In [5]:
pd.Series(["Has relevant experience", "No relevant experience"]).astype("bool")

0    True
1    True
dtype: bool

Both become True, so 5,366 "No" answers would turn into "Yes".

Instead we compare every row with the text "Has relevant experience". The comparison gives True where the text matches and False everywhere else, which is exactly the bool column we want.

In [6]:
df["relevant_experience"] = df["relevant_experience"] == "Has relevant experience"

## 5. Ordered categories

For ordinal columns `astype("category")` isn't enough, because pandas doesn't know any order. We have to give it the order ourselves as a list, from lowest to highest:

- **enrolled_university:** no_enrollment < Part time course < Full time course
- **education_level:** Primary School < High School < Graduate < Masters < Phd
- **experience:** <1 < 1 < 2 < ... < 20 < >20
- **company_size:** <10 < 10-49 < 50-99 < 100-499 < 500-999 < 1000-4999 < 5000-9999 < 10000+
- **last_new_job:** never < 1 < 2 < 3 < 4 < >4

Putting "never" at the bottom is our decision. It could also be read as the longest time, so we should be able to explain it in the presentation.

The experience list has 22 values. Instead of typing them, the code builds it: `range(1, 21)` gives the numbers 1 to 20, and `str(i)` turns them into text, because the CSV stores them as text.

`pd.CategoricalDtype(categories=..., ordered=True)` describes the new type: these categories, in this order. The loop applies it to each of the five columns.

**Careful:** a value that is not in the list silently becomes NaN, without an error. A typo like "PhD" instead of "Phd" would delete 414 values. That's why the validation in section 6 compares the missing values.

In [7]:
orders = {
    "enrolled_university": ["no_enrollment", "Part time course", "Full time course"],
    "education_level": ["Primary School", "High School", "Graduate", "Masters", "Phd"],
    "experience": ["<1"] + [str(i) for i in range(1, 21)] + [">20"],
    "company_size": ["<10", "10-49", "50-99", "100-499", "500-999", "1000-4999", "5000-9999", "10000+"],
    "last_new_job": ["never", "1", "2", "3", "4", ">4"],
}

for col, order in orders.items():
    df[col] = df[col].astype(pd.CategoricalDtype(categories=order, ordered=True))

Now every column has its new type. Note that `df.dtypes` shows ordered categories as plain `category`, it doesn't show the order.

In [8]:
print("New data types:")
print(df.dtypes)

New data types:
student_id                   int32
city                      category
city_development_index     float16
gender                    category
relevant_experience           bool
enrolled_university       category
education_level           category
major_discipline          category
experience                category
company_size              category
company_type              category
last_new_job              category
training_hours               int32
job_change                    bool
dtype: object


To see the order, look at the dtype of a single column. `ordered=True` and the list of categories show that the order is stored.

In [9]:
df["company_size"].dtype

CategoricalDtype(categories=['<10', '10-49', '50-99', '100-499', '500-999', '1000-4999',
                  '5000-9999', '10000+'],
, ordered=True, categories_dtype=object)

## 6. Validation: did the conversion work?

Here we check if the data is still complete and if the values make sense. Two ideas behind it:

- **Allowed values:** categories may only contain values from a fixed list. "Male", "Female" and "Other" are fine for gender, "male" or "Mael" would be errors.
- **Plausible ranges:** numbers must be in a realistic range. The city development index is an index, so it has to be between 0 and 1. Training hours can't be negative.

The most important check is the first one: since unknown values silently turn into NaN, the number of missing values must be exactly the same as before the conversion.

`assert` checks a condition. If it's true nothing happens, if it's false the cell stops with an `AssertionError` at exactly that line. If everything passes, a green message with the memory saving is printed.

| Check | Expected |
|---|---|
| Rows / columns | 19,158 / 14 |
| Missing values | 20,733 in total, same per column as before |
| gender, major_discipline, company_type | only the known categories |
| city | 123 cities, all in the form city_number |
| relevant_experience | 13,792 True |
| job_change | 4,777 True |
| city_development_index | between 0 and 1 |
| training_hours | 1 to 336 |
| student_id | 1 to 33,380, all unique |
| company_size >= "1000-4999" | 3,910 rows (order works) |
| education_level >= "Masters" | 4,775 rows (order works) |
| Memory | less than before |

The fixed numbers come from this dataset. With a different CSV these checks would fail on purpose.

In [10]:
##### !!!! ENTIRELY WRITTEN BY CLAUDE !!!! #####
# ==> maybe still useful in presentation, because he told us to use as much AI as we wanted an it shows, that we have high standards

try:
    assert df.shape == (19158, 14)

    # values outside the category lists turn into NaN, so the counts must not change
    assert df.isna().sum().equals(missing_before)
    assert df.isna().sum().sum() == 20733

    # allowed values
    allowed = {
        "gender": {"Male", "Female", "Other"},
        "major_discipline": {"STEM", "Humanities", "Business Degree", "Arts", "No Major", "Other"},
        "company_type": {"Pvt Ltd", "Funded Startup", "Public Sector", "Early Stage Startup", "NGO", "Other"},
    }

    for col, values in allowed.items():
        assert set(df[col].dropna()) <= values, col

    assert df["city"].str.fullmatch(r"city_\d+").all()
    assert df["city"].nunique() == 123
    assert df["relevant_experience"].sum() == 13792
    assert df["job_change"].sum() == 4777

    # plausible ranges
    assert df["city_development_index"].between(0, 1).all()
    assert df["training_hours"].min() == 1 and df["training_hours"].max() == 336
    assert df["student_id"].min() == 1 and df["student_id"].max() == 33380
    assert df["student_id"].is_unique

    # order works
    assert all(df[col].cat.ordered for col in orders)
    assert (df["company_size"] >= "1000-4999").sum() == 3910
    assert (df["education_level"] >= "Masters").sum() == 4775

    memory_after = df.memory_usage(deep=True).sum()
    assert memory_after < memory_before

    print("\033[32m\nAll checks passed, memory: "
          f"{memory_before / 1e6:.1f} MB -> {memory_after / 1e6:.2f} MB\033[0m")
except AssertionError:
    print("\033[31m\nChecks failed\033[0m")
    raise
except Exception:
    print("\033[31m\nChecks failed\033[0m")
    raise


All checks passed, memory: 11.0 MB -> 0.42 MB


### Where does the memory saving come from?

This table compares every column before and after. The original file is read again just for the comparison, `df` stays as it is.

In [11]:
original = pd.read_csv("../customer_train.csv")

memory_per_column = pd.DataFrame({
    "type before": original.dtypes,
    "type after": df.dtypes,
    "KB before": original.memory_usage(deep=True, index=False) / 1e3,
    "KB after": df.memory_usage(deep=True, index=False) / 1e3,
}).round(1).sort_values("KB before", ascending=False)

memory_per_column

,type before,type after,KB before,KB after
relevant_experience,object,bool,1374.0,19.2
enrolled_university,object,category,1191.1,19.5
city,object,category,1082.6,30.3
education_level,object,category,1082.0,19.6
experience,object,category,969.2,20.8
major_discipline,object,category,965.2,19.7
last_new_job,object,category,963.4,19.6
company_type,object,category,943.1,19.7
gender,object,category,923.4,19.4
company_size,object,category,917.8,19.9


Almost the whole saving comes from the ten text columns. The number columns only shrink from 8 to 4 or 2 bytes per row, so they add little.

`category` only pays off when few values repeat often. For `student_id` it would be a bad idea, because every ID appears only once.

## 7. Data problems

The validation checked every column on its own. Now we look at whole rows and at the distribution: values that are valid on their own, but don't fit together or stand out.

Everything here is only counted, nothing gets changed. **Clear** means it can't be true, **soft** means it's unusual but possible.

### Logical breaks

| Problem | Type | How we check it |
|---|---|---|
| Last job change longer ago than total experience | clear | compare last_new_job and experience |
| Major listed but only school education | clear | education_level at most High School, major_discipline filled and not "No Major" |
| Primary School but enrolled at a university | clear | education_level is Primary School, enrolled_university is a course |
| Graduate or higher without a major | soft | education_level at least Graduate, major_discipline empty or "No Major" |
| Under 1 year of experience but relevant experience | soft | experience is <1, relevant_experience is True |
| Over 20 years of experience, never changed jobs | soft | experience is >20, last_new_job is never |
| Only one of company_size and company_type missing | soft | one is empty, the other isn't |
| City with more than one development index | clear | group by city, count the different index values |

To compare experience and last_new_job we use `cat.codes`. Every category gets a number following its order: for experience "<1" is 0, "1" is 1 and so on up to ">20" as 21. For last_new_job "never" is 0 and ">4" is 5. Missing values get -1. Because both orders count years, the numbers can be compared directly.

Each check gives one True or False per row. `.sum()` counts the True values, `.mean()` gives their share.

In [12]:
print("Logical breaks:")

experience = df["experience"].cat.codes
last_job = df["last_new_job"].cat.codes

school_only = df["education_level"] <= "High School"
no_major = df["major_discipline"].isna() | (df["major_discipline"] == "No Major")
in_course = df["enrolled_university"].isin(["Part time course", "Full time course"])

logical_breaks = {
    "last job change longer ago than experience (clear)": (experience >= 0) & (last_job > experience),
    "major but only school education (clear)": school_only & ~no_major,
    "primary school but enrolled at university (clear)": (df["education_level"] == "Primary School") & in_course,
    "graduate or higher without major (soft)": (df["education_level"] >= "Graduate") & no_major,
    "under 1 year but relevant experience (soft)": (df["experience"] == "<1") & df["relevant_experience"],
    "over 20 years, never changed jobs (soft)": (df["experience"] == ">20") & (df["last_new_job"] == "never"),
    "only one company column missing (soft)": df["company_size"].isna() != df["company_type"].isna(),
}

for name, rows in logical_breaks.items():
    print(f"{name}: {rows.sum():,} ({rows.mean():.1%})")

index_per_city = df.groupby("city", observed=True)["city_development_index"].nunique()
print(f"cities with more than one index (clear): {(index_per_city > 1).sum()}")

Logical breaks:
last job change longer ago than experience (clear): 350 (1.8%)
major but only school education (clear): 0 (0.0%)
primary school but enrolled at university (clear): 44 (0.2%)
graduate or higher without major (soft): 251 (1.3%)
under 1 year but relevant experience (soft): 164 (0.9%)
over 20 years, never changed jobs (soft): 126 (0.7%)
only one company column missing (soft): 1,358 (7.1%)
cities with more than one index (clear): 0


### Duplicates

Every student_id is unique, but that alone doesn't prove there are no duplicates. So we leave out the ID and look for rows that are identical in all other columns.

| Problem | How we check it |
|---|---|
| Same row with a different student_id | `duplicated()` without student_id |
| Same profile but different job_change | `duplicated()` without student_id and job_change, minus the identical rows |

`duplicated()` marks every row that already appeared further up. Identical rows can also be two different people with the same profile, so a match alone doesn't prove an error.

In [13]:
print("Duplicates:")
identical = df.drop(columns="student_id").duplicated().sum()
same_profile = df.drop(columns=["student_id", "job_change"]).duplicated().sum()
print(f"identical rows without student_id: {identical}")
print(f"same profile, different job_change: {same_profile - identical}")

Duplicates:
identical rows without student_id: 49
same profile, different job_change: 25


### Outliers

Only training_hours and city_development_index are real numbers, so we check those two with the **IQR rule**:

- Q1 is the value below which 25 % of the data lies, Q3 the value below which 75 % lies
- IQR = Q3 - Q1 is the range of the middle half of the data
- everything below Q1 - 1.5 × IQR or above Q3 + 1.5 × IQR counts as an outlier

An outlier is not automatically wrong. It's unusual but possible, so we look at them before removing anything.

In [14]:
print("Outliers (IQR rule):")
for col in ["training_hours", "city_development_index"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = ~df[col].between(low, high)
    print(f"{col}: {outliers.sum():,} ({outliers.mean():.1%}) outside {low:.3f} to {high:.3f}")

Outliers (IQR rule):
training_hours: 984 (5.1%) outside -74.500 to 185.500
city_development_index: 17 (0.1%) outside 0.471 to 1.189


### Hidden missing values

Sometimes missing values aren't empty but hidden behind a placeholder, e.g. "n/a", "-" or "unknown". pandas only recognizes empty cells as missing, so we compare the categories of all text columns with a list of common placeholders. The check ignores upper and lower case and spaces around the text.

We also count "Other", since it can be used instead of a real answer.

| Problem | How we check it |
|---|---|
| Placeholders like "n/a", "-", "unknown" | compare all categories with a list of placeholders |
| "Other" used instead of a real answer | count "Other" in gender, major_discipline and company_type |

In [15]:
print("Hidden missing values:")
placeholders = {"", ".", "-", "?", "n/a", "na", "nan", "none", "null", "unknown"}
for col in df.select_dtypes("category"):
    found = [c for c in df[col].cat.categories if c.strip().lower() in placeholders]
    if found:
        print(f"{col}: {found}")
print("placeholders checked in all text columns")

for col in ["gender", "major_discipline", "company_type"]:
    other = df[col] == "Other"
    print(f"{col} = Other: {other.sum():,} ({other.mean():.1%})")

Hidden missing values:
placeholders checked in all text columns
gender = Other: 191 (1.0%)
major_discipline = Other: 381 (2.0%)
company_type = Other: 121 (0.6%)


### Reading the results

- **Last job change vs. experience:** "clear" is too strong here. "<1" year of experience with last_new_job = 1 also counts, which can just be rounding. We also don't know for sure that last_new_job measures the time since the last change. Better call it a possible contradiction.
- **Primary School at a university:** not clearly wrong either, e.g. if the training course itself was counted as enrollment.
- **training_hours:** all outliers are high values, caused by the skewed distribution. They're possible, so we keep them.
- **Duplicates:** they can be different people with the same profile, so we keep them and only mention them.

How we handle the findings is decided in the next step.

## 8. Saving the converted data

The converted table is saved as a new file in `week 2`. The original `customer_train.csv` is only read in this notebook and never written to, so it stays as it is.

Why not as CSV? A CSV file only stores text. When it's read again, all types are gone and pandas guesses `object`, `int64` and `float64` again. A **pickle** file stores the DataFrame together with its types, so the next notebook can continue right away. (Parquet would work as well, but needs the extra package pyarrow.)

Only open pickle files you created yourself or got from someone you trust, because loading one can run code.

In [16]:
df.to_pickle("customer_train_typed.pkl")

Finally we load the new file and compare: the saved data must have the new types, the original CSV still has the old ones.

In [17]:
saved = pd.read_pickle("customer_train_typed.pkl")
original = pd.read_csv("../customer_train.csv")

assert saved.equals(df)
assert saved.dtypes.equals(df.dtypes)

pd.DataFrame({"original CSV": original.dtypes, "saved file": saved.dtypes})

,original CSV,saved file
student_id,int64,int32
city,object,category
city_development_index,float64,float16
gender,object,category
relevant_experience,object,bool
enrolled_university,object,category
education_level,object,category
major_discipline,object,category
experience,object,category
company_size,object,category


To use the converted data in another notebook in `week 2`:

```python
df = pd.read_pickle("customer_train_typed.pkl")
```